In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df_emp =  spark.read.format('csv').option('header',True).load('/Volumes/workspace/default/datasets/emp_data_ABC/*.csv')
df_emp.display()

In [0]:
%sql
CREATE TABLE IF NOT EXISTS EMP_ABC (
emp_id BIGINT ,
emp_name STRING,
emp_sal BIGINT,
year STRING,
create_ts TIMESTAMP,
lst_updt_ts TIMESTAMP
) using delta

In [0]:
df_emp.createOrReplaceTempView('emp_view')

In [0]:
%sql
MERGE into emp_abc
using emp_view
on emp_abc.emp_id = emp_view.emp_id
when MATCHED AND emp_view.emp_sal != emp_abc.emp_sal 
THEN UPDATE
SET
emp_abc.emp_sal=emp_view.emp_sal,
emp_abc.year = emp_view.year,
emp_abc.lst_updt_ts = current_timestamp()
when NOT MATCHED THEN
INSERT (emp_id,emp_name,emp_sal,year,create_ts,lst_updt_ts)
VALUES(emp_view.emp_id,emp_view.emp_name,emp_view.emp_sal,emp_view.year,current_timestamp(),current_timestamp())

In [0]:
%sql
--select * from emp_abc

In [0]:
%sql
select * from emp_abc

## Merge using Pyspark

In [0]:
from delta.tables import DeltaTable

In [0]:
df_emp_target = DeltaTable.forName(spark,"workspace.default.emp_abc")

In [0]:
df_emp_target.alias("tgt").merge(
    source=df_emp.alias("src"),
    condition="tgt.emp_id = src.emp_id"
).whenMatchedUpdate(
    condition="tgt.emp_sal != src.emp_sal",
    set={
        "emp_sal": "src.emp_sal",
        "year": "src.year",
        "lst_updt_ts": "current_timestamp()"
    }
).whenNotMatchedInsert(
    values={
        "emp_id": "src.emp_id",
        "emp_name": "src.emp_name",
        "emp_sal": "src.emp_sal",
        "year": "src.year",
        "create_ts": "current_timestamp()",
        "lst_updt_ts": "current_timestamp()"
    }
).execute()